In [1]:
import os
import xml.etree.ElementTree as ET
import pandas as pd

In [2]:
input_dir = "E:\school(softwarica)\Sem-4\BigDataProject\Data\TimeTable"
output_path = "E:\school(softwarica)\Sem-4\BigDataProject\Data\FinalCSV\clean_timetabledata.csv"
os.makedirs("E:\school(softwarica)\Sem-4\BigDataProject\Data\FinalCSV", exist_ok = True)

In [3]:
ns = {'txc' : 'http://www.transxchange.org.uk/'}
data = []

In [4]:
for filename in os.listdir(input_dir):
    if filename.endswith(".xml"):
        tree = ET.parse(os.path.join(input_dir, filename))
        root = tree.getroot()

        # mapping StopPointsRefs to CommonNames
        stop_map = {}
        for stop in root.findall('.//txc:AnnotatedStopPointRef', ns):
            ref = stop.find('txc:StopPointRef', ns)
            name = stop.find('txc:CommonName', ns)
            if ref is not None and name is not None:
                stop_map[ref.text] = name.text

        # mapping TimingLink Ids to StopPointRefs
        link_map = {}
        for link in root.findall('.//txc:JourneyPatternTimingLink', ns):
            link_id = link.get('id')
            stop_ref = link.find('.//txc:To/txc:StopPointRef', ns)
            if link_id and stop_ref is not None:
                link_map[link_id] = stop_ref.text

        # mapping line id 
        line_lookup = {}
        for line_el in root.findall('.//txc:Services/txc:Service/txc:Lines/txc:Line', ns):
            line_id = line_el.get('id')
            name_el = line_el.find('txc:LineName', ns)
            line_lookup[line_id] = name_el.text if name_el is not None else "N/A"

        service = root.find('.//txc:ServiceCode', ns)

        # Extracting journeys to resolve LineName per journey
        for journey in root.findall('.//txc:VehicleJourney', ns):
            j_code = journey.find('txc:VehicleJourneyCode', ns)
            j_text = j_code.text if j_code is not None else "N/A"

            vj_line_ref = journey.find('txc:LineRef', ns)
            line_name = line_lookup.get(vj_line_ref.text, "N/A") if vj_line_ref is not None else "N/A"

            for v_link in journey.findall('.//txc:VehicleJourneyTimingLink', ns):
                ref = v_link.find('txc:JourneyPatternTimingLinkRef', ns)
                run_time = v_link.find('txc:RunTime', ns)

                stop_id = link_map.get(ref.text, "N/A") if ref is not None else "N/A"

                data.append({
                    "ServiceCode": service.text if service is not None else "N/A",
                    "LineName": line_name,
                    "JourneyCode": j_text,
                    "StopPointRef": stop_id,
                    "StopName": stop_map.get(stop_id, "N/A"),
                    "RunTime": run_time.text if run_time is not None else "N/A",
                    "Source": filename
                })

In [5]:
#saving output
df = pd.DataFrame(data)
df.to_csv(output_path, index = False)
print(f"Extraction complete. Total records: {len(df)}")
print(f"Distinct lines: {df['LineName'].nunique()}")

Extraction complete. Total records: 176615
Distinct lines: 42


In [6]:
check = pd.read_csv('E:\school(softwarica)\Sem-4\BigDataProject\Data\FinalCSV\clean_timetabledata.csv')
print(check['LineName'].nunique())

42


In [7]:
check.head()

,ServiceCode,LineName,JourneyCode,StopPointRef,StopName,RunTime,Source
0,PH0005863:12,1,VJ924,340001219CNR,Blackbird Leys Sandy Lane,PT1M0S,1-None--SCOX-OXHR-2024-01-01-OX_26-05-31_[East...
1,PH0005863:12,1,VJ924,340001234CNR,Blackbird Leys Wingate Close,PT0M0S,1-None--SCOX-OXHR-2024-01-01-OX_26-05-31_[East...
2,PH0005863:12,1,VJ924,340001229CNR,Blackbird Leys Longlands Road,PT2M0S,1-None--SCOX-OXHR-2024-01-01-OX_26-05-31_[East...
3,PH0005863:12,1,VJ924,340001224CNR,Blackbird Leys Pegasus Court,PT1M0S,1-None--SCOX-OXHR-2024-01-01-OX_26-05-31_[East...
4,PH0005863:12,1,VJ924,340001231CNR,Blackbird Leys Rampion Close,PT1M0S,1-None--SCOX-OXHR-2024-01-01-OX_26-05-31_[East...
